# 85) Colab Adapter Training (T4)

This notebook trains a LoRA adapter on a free Colab T4 GPU using a self-contained training loop.
It minimizes reliance on repo scripts and uses the calibrated T4 config defaults.

Optional: mount Drive for persistence; otherwise artifacts are stored under `/content`.

After download, place `lora_adapter` into `artifacts/runs/finetune_unsloth/lora_adapter` in your local repo.

In [1]:
# 0) Optional: mount Google Drive for persistence
import importlib.util
import os

DRIVE_MOUNTED = False
is_kaggle = bool(os.getenv("KAGGLE_URL_BASE")) or os.path.exists("/kaggle")
is_colab = importlib.util.find_spec("google.colab") is not None

if is_colab and not is_kaggle:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("Drive mount unsupported in this runtime; continuing without Drive.")
else:
    print("Drive mount skipped (Kaggle or non-Colab runtime).")

Drive mount skipped (Kaggle or non-Colab runtime).


In [2]:
# 1) Confirm GPU runtime (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

Sat Apr 18 08:13:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 1.2) Clear stale GPU processes (skip current kernel)
import os
import subprocess

current_pid = os.getpid()
result = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
pids = [int(line.strip()) for line in result.stdout.splitlines() if line.strip().isdigit()]
stale = [pid for pid in pids if pid != current_pid]

if not pids:
    print("No GPU processes found.")
elif not stale:
    print(f"GPU process matches current kernel pid: {current_pid}. Nothing to kill.")
else:
    for pid in stale:
        try:
            os.kill(pid, 9)
            print(f"Killed GPU process pid={pid}")
        except Exception as exc:
            print(f"Failed to kill pid={pid}: {exc}")

No GPU processes found.


In [4]:
# 1.5) Optional: set HF_TOKEN for gated models
import os
import getpass

if not os.getenv("HF_TOKEN"):
    token = getpass.getpass("HF token (press Enter to skip): ")
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN set for this session.")
    else:
        print("No token set.")
else:
    print("HF_TOKEN already set in environment.")

HF_TOKEN set for this session.


In [15]:
# 2) Clone repo + move into it
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/aaliyan1230/polyglot-grounded-qa.git"
BRANCH = "feat/kg-rag-abstention"
IS_KAGGLE = bool(os.getenv("KAGGLE_URL_BASE")) or Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path("/content")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_ROOT = WORK_ROOT / "polyglot-grounded-qa"
if not REPO_ROOT.exists():
    !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}

%cd {REPO_ROOT}

tracked_finetune_paths = [
    Path("data/benchmarks/finetune/sft_dataset.jsonl"),
    Path("data/benchmarks/finetune/public_sft_dataset.jsonl"),
    Path("data/benchmarks/finetune/sft_dataset_merged.jsonl"),
    Path("data/benchmarks/finetune/train.jsonl"),
    Path("data/benchmarks/finetune/val.jsonl"),
    Path("data/benchmarks/finetune/test.jsonl"),
    Path("data/benchmarks/finetune/formatted/train.text.jsonl"),
    Path("data/benchmarks/finetune/formatted/val.text.jsonl"),
    Path("data/benchmarks/finetune/formatted/test.text.jsonl"),
]

subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
for rel_path in tracked_finetune_paths:
    abs_path = REPO_ROOT / rel_path
    if not abs_path.exists():
        continue
    tracked_locally = subprocess.run(
        ["git", "ls-files", "--error-unmatch", str(rel_path)],
        cwd=REPO_ROOT,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    ).returncode == 0
    if not tracked_locally:
        abs_path.unlink()
        print("Removed stale local finetune file before pull:", rel_path)

subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
print("WORK_ROOT:", WORK_ROOT)
print("REPO_ROOT:", REPO_ROOT)

/kaggle/working/polyglot-grounded-qa


From https://github.com/aaliyan1230/polyglot-grounded-qa
 * branch            feat/kg-rag-abstention -> FETCH_HEAD
   7200a54..61c9e82  feat/kg-rag-abstention -> origin/feat/kg-rag-abstention


Updating 7200a54..61c9e82
Fast-forward
 scripts/generate_tuned_predictions.py | 5 ++++-
 scripts/run_trained_adapter_eval.py   | 2 ++
 2 files changed, 6 insertions(+), 1 deletion(-)
WORK_ROOT: /kaggle/working
REPO_ROOT: /kaggle/working/polyglot-grounded-qa


From https://github.com/aaliyan1230/polyglot-grounded-qa
 * branch            feat/kg-rag-abstention -> FETCH_HEAD


In [6]:
# 3) Configure export paths (Drive optional)
from pathlib import Path
import os

IS_KAGGLE = bool(os.getenv("KAGGLE_URL_BASE")) or Path("/kaggle/working").exists()
WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working") if IS_KAGGLE else Path("/content"))
REPO_ROOT = REPO_ROOT if "REPO_ROOT" in globals() else WORK_ROOT / "polyglot-grounded-qa"
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_MYDRIVE = DRIVE_MOUNT_POINT / "MyDrive"

drive_ready = bool(globals().get("DRIVE_MOUNTED", False)) and DRIVE_MYDRIVE.exists()
if drive_ready:
    DRIVE_ROOT = DRIVE_MYDRIVE / "polyglot_grounded_qa"
else:
    DRIVE_ROOT = WORK_ROOT / "polyglot_grounded_qa_exports"

DRIVE_RUN_DIR = DRIVE_ROOT / "finetune_unsloth"
DRIVE_ARTIFACTS_DIR = DRIVE_ROOT / "exports"

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT:", WORK_ROOT)
print("REPO_ROOT:", REPO_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("DRIVE_RUN_DIR:", DRIVE_RUN_DIR)
print("DRIVE_ARTIFACTS_DIR:", DRIVE_ARTIFACTS_DIR)

WORK_ROOT: /kaggle/working
REPO_ROOT: /kaggle/working/polyglot-grounded-qa
DRIVE_ROOT: /kaggle/working/polyglot_grounded_qa_exports
DRIVE_RUN_DIR: /kaggle/working/polyglot_grounded_qa_exports/finetune_unsloth
DRIVE_ARTIFACTS_DIR: /kaggle/working/polyglot_grounded_qa_exports/exports


In [7]:
# 4) Install stable training dependencies (single clean set)
import os
import sys

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["DISABLE_TRANSFORMERS_AV"] = "1"

print("Python:", sys.version)
print("Executable:", sys.executable)

%pip install -U pip setuptools wheel packaging
%pip uninstall -y torchvision torchaudio tensorflow keras opencv-python opencv-python-headless opencv-contrib-python
%pip install -U pyyaml polars duckdb
%pip install -U "transformers==4.46.3" "trl==0.12.2" "peft==0.14.0" "datasets==2.21.0" "accelerate==0.34.2" bitsandbytes

import importlib
for m in ["numpy", "scipy", "pandas", "pyarrow", "transformers", "trl", "peft", "datasets", "accelerate", "torch", "bitsandbytes"]:
    try:
        mod = importlib.import_module(m)
        print(m, getattr(mod, "__version__", ""))
    except Exception as exc:
        print(f"WARN: failed to import {m}: {exc}")

print("Dependency installation complete.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Executable: /usr/bin/python3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requ

In [8]:
# 5) Ensure formatted SFT files (self-contained)
from pathlib import Path
import json
import math
import random
import shutil
from collections import defaultdict

REPO_ROOT = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
data_dir = REPO_ROOT / "data" / "benchmarks" / "finetune"
formatted_dir = data_dir / "formatted"

train_path = data_dir / "train.jsonl"
val_path = data_dir / "val.jsonl"
test_path = data_dir / "test.jsonl"

formatted_train = formatted_dir / "train.text.jsonl"
formatted_val = formatted_dir / "val.text.jsonl"
formatted_test = formatted_dir / "test.text.jsonl"

def _read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def _write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=True) + "\n")

def _split_bucket(rows: list[dict], train_ratio: float, val_ratio: float):
    n = len(rows)
    train_n = round(n * train_ratio)
    val_n = round(n * val_ratio)
    if n >= 4 and val_n == 0:
        val_n = 1
    test_n = n - train_n - val_n
    if n >= 4 and test_n == 0:
        test_n = 1
        train_n = max(0, n - val_n - test_n)
    train_n = max(0, min(train_n, n))
    val_n = max(0, min(val_n, n - train_n))
    return rows[:train_n], rows[train_n : train_n + val_n], rows[train_n + val_n :]

def _abstain_ratio(rows: list[dict]) -> float:
    if not rows:
        return 0.0
    abstained = sum(1 for row in rows if bool(row["target"].get("abstained", False)))
    return abstained / len(rows)

def _rebalance_abstain_ratio(donor: list[dict], recipient: list[dict], min_ratio: float) -> None:
    if not recipient:
        return
    recipient_abstained = sum(1 for row in recipient if bool(row["target"].get("abstained", False)))
    needed = math.ceil(min_ratio * len(recipient)) - recipient_abstained
    if needed <= 0:
        return
    donor_abstained_idx = [
        idx for idx, row in enumerate(donor) if bool(row["target"].get("abstained", False))
    ]
    recipient_non_abstained_idx = [
        idx for idx, row in enumerate(recipient) if not bool(row["target"].get("abstained", False))
    ]
    swaps = min(needed, len(donor_abstained_idx), len(recipient_non_abstained_idx))
    for i in range(swaps):
        d_idx = donor_abstained_idx[i]
        r_idx = recipient_non_abstained_idx[i]
        donor[d_idx], recipient[r_idx] = recipient[r_idx], donor[d_idx]

# Try to sync from Drive if repo data files are missing.
drive_data_dir = None
if "DRIVE_ROOT" in globals():
    drive_data_dir = Path(DRIVE_ROOT) / "data" / "benchmarks" / "finetune"
    if drive_data_dir.exists():
        for name in [
            "train.jsonl",
            "val.jsonl",
            "test.jsonl",
            "sft_dataset_merged.jsonl",
            "sft_dataset.jsonl",
            "public_sft_dataset.jsonl",
        ]:
            src = drive_data_dir / name
            dst = data_dir / name
            if src.exists() and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
                print("Copied from Drive:", src, "->", dst)

source_candidates = [
    data_dir / "sft_dataset_merged.jsonl",
    data_dir / "sft_dataset.jsonl",
    data_dir / "public_sft_dataset.jsonl",
]
source_path = next((p for p in source_candidates if p.exists()), None)

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    if source_path is None:
        print("No dataset found; generating a minimal seed dataset for quick training.")
        seed_chunks = [
            {
                "doc_id": "seed-doc-1",
                "chunk_id": "seed-chunk-1",
                "text": "Polyglot grounded QA uses retrieval with citations and verification.",
            },
            {
                "doc_id": "seed-doc-2",
                "chunk_id": "seed-chunk-2",
                "text": "Language packs inherit base behavior and only override locale-specific rules.",
            },
        ]
        queries = [
            "What is grounded QA?",
            "How does language-pack inheritance work?",
            "Why are citations required?",
        ]
        answer_map = {
            "What is grounded QA?": ("Polyglot grounded QA uses retrieval with citations and verification.", ["seed-chunk-1"]),
            "How does language-pack inheritance work?": ("Language packs inherit base behavior and only override locale-specific rules.", ["seed-chunk-2"]),
            "Why are citations required?": ("Grounded QA answers with explicit evidence citations.", ["seed-chunk-1"]),
        }
        languages = ["base", "es", "es-MX", "fr", "tr"]
        rows: list[dict] = []
        counter = 0
        for language in languages:
            for query in queries:
                answer, citations = answer_map[query]
                rows.append({
                    "id": f"sft-{counter:06d}",
                    "language": language,
                    "query": query,
                    "retrieved_chunks": seed_chunks,
                    "target": {
                        "answer": answer,
                        "citations": citations,
                        "abstained": False,
                        "reason": "",
                    },
                    "label_type": "answerable",
                    "source": "seed-minimal",
                })
                counter += 1
                rows.append({
                    "id": f"sft-{counter:06d}",
                    "language": language,
                    "query": f"{query} [insufficient-evidence]",
                    "retrieved_chunks": [],
                    "target": {
                        "answer": "I do not have enough evidence.",
                        "citations": [],
                        "abstained": True,
                        "reason": "insufficient_evidence",
                    },
                    "label_type": "insufficient_evidence",
                    "source": "seed-minimal",
                })
                counter += 1
        source_path = data_dir / "sft_dataset.jsonl"
        _write_jsonl(source_path, rows)
        print("Wrote seed dataset to", source_path)
    rows = _read_jsonl(source_path)
    if not rows:
        raise ValueError(f"{source_path} is empty.")
    rng = random.Random(42)
    by_language: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        language = str(row.get("language", "base"))
        by_language[language].append(row)
    train_rows: list[dict] = []
    val_rows: list[dict] = []
    test_rows: list[dict] = []
    for _, bucket in by_language.items():
        rng.shuffle(bucket)
        train, val, test = _split_bucket(bucket, train_ratio=0.8, val_ratio=0.1)
        train_rows.extend(train)
        val_rows.extend(val)
        test_rows.extend(test)
    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    rng.shuffle(test_rows)
    _rebalance_abstain_ratio(train_rows, val_rows, min_ratio=0.2)
    _rebalance_abstain_ratio(train_rows, test_rows, min_ratio=0.2)
    _write_jsonl(train_path, train_rows)
    _write_jsonl(val_path, val_rows)
    _write_jsonl(test_path, test_rows)
    print("Wrote splits from", source_path)

def _retrieval_block(chunks: list[dict]) -> str:
    if not chunks:
        return "[NO_RETRIEVED_EVIDENCE]"
    lines = []
    for chunk in chunks:
        chunk_id = str(chunk.get("chunk_id", ""))
        text = str(chunk.get("text", "")).strip()
        lines.append(f"[{chunk_id}] {text}")
    return "\n".join(lines)

def _target_json(row: dict) -> str:
    target = row.get("target", {})
    payload = {
        "answer": target.get("answer", ""),
        "citations": target.get("citations", []),
        "abstained": bool(target.get("abstained", False)),
        "reason": target.get("reason", ""),
    }
    return json.dumps(payload, ensure_ascii=True)

def _instruction(row: dict) -> str:
    language = str(row.get("language", "base"))
    query = str(row.get("query", "")).strip()
    evidence = _retrieval_block(row.get("retrieved_chunks", []))
    return (
        "You are a grounded QA model. Use only evidence below.\n"
        "If evidence is insufficient, abstain.\n"
        "Return strict JSON with keys: answer, citations, abstained, reason.\n\n"
        f"Language: {language}\n"
        f"Query: {query}\n"
        f"Evidence:\n{evidence}"
    )

def _build_text_record(row: dict) -> dict:
    prompt = _instruction(row)
    completion = _target_json(row)
    return {
        "id": row.get("id", ""),
        "language": row.get("language", "base"),
        "text": f"<|user|>\n{prompt}\n<|assistant|>\n{completion}",
        "source": row.get("source", ""),
        "label_type": row.get("label_type", ""),
    }

if formatted_train.exists() and formatted_val.exists() and formatted_test.exists():
    print("Formatted files already exist.")
else:
    for split, path in [("train", train_path), ("val", val_path), ("test", test_path)]:
        rows = _read_jsonl(path)
        if not rows:
            raise ValueError(f"{path} is empty.")
        out = [_build_text_record(row) for row in rows]
        _write_jsonl(formatted_dir / f"{split}.text.jsonl", out)
    print("Wrote formatted text files to", formatted_dir)

train_rows = _read_jsonl(train_path)
val_rows = _read_jsonl(val_path)
test_rows = _read_jsonl(test_path)
print("Data prep OK.")
print("Train:", len(train_rows), "Val:", len(val_rows), "Test:", len(test_rows))
print("Formatted train path:", formatted_train)

Formatted files already exist.
Data prep OK.
Train: 4322 Val: 540 Test: 539
Formatted train path: /kaggle/working/polyglot-grounded-qa/data/benchmarks/finetune/formatted/train.text.jsonl


In [9]:
# 5.5) Pull pre-trained adapter from Kaggle dataset OR train fresh
# ── Set SKIP_TRAINING = True to reuse a previously-published adapter ──
# ── Set SKIP_TRAINING = False to do a fresh training run (cells 11-14) ──
# NOTE: The dataset was scaled up to ~4300 train / 540 val / 539 test rows
#       (real XQuAD data). Set to False to retrain on the new data.
SKIP_TRAINING = True  # Adapter already trained; re-running eval only with generation fix

if SKIP_TRAINING:
    from pathlib import Path
    import shutil
    import subprocess

    DATASET_ID = "aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts"
    DOWNLOAD_DIR = Path("/tmp/kaggle_adapter_pull")

    if DOWNLOAD_DIR.exists():
        shutil.rmtree(DOWNLOAD_DIR)
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    print("Downloading adapter from Kaggle dataset:", DATASET_ID)
    cp = subprocess.run(
        ["kaggle", "datasets", "download", DATASET_ID, "-p", str(DOWNLOAD_DIR), "--unzip"],
        text=True, capture_output=True,
    )
    print(cp.stdout.strip())
    if cp.stderr.strip():
        print(cp.stderr.strip())
    if cp.returncode != 0:
        raise RuntimeError("Kaggle download failed. Check KAGGLE_USERNAME / KAGGLE_KEY in Secrets.")

    # Locate the adapter inside the downloaded tree
    adapter_src = DOWNLOAD_DIR / "runs" / "finetune_unsloth" / "lora_adapter"
    if not adapter_src.exists():
        adapter_src = DOWNLOAD_DIR / "artifacts" / "runs" / "finetune_unsloth" / "lora_adapter"
    if not adapter_src.exists():
        raise FileNotFoundError(
            f"Could not find lora_adapter in download. Contents: {list(DOWNLOAD_DIR.rglob('*'))[:30]}"
        )

    # Place adapter where downstream cells expect it
    if "DRIVE_RUN_DIR" in globals():
        OUTPUT_DIR = Path(DRIVE_RUN_DIR)
    else:
        OUTPUT_DIR = Path("/kaggle/working/polyglot_grounded_qa_exports/finetune_unsloth")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    adapter_dst = OUTPUT_DIR / "lora_adapter"
    if adapter_dst.exists():
        shutil.rmtree(adapter_dst)
    shutil.copytree(adapter_src, adapter_dst)

    # Restore eval artifacts if present
    for name in [
        "raw_model_predictions_adapter.jsonl",
        "tuned_predictions_adapter.jsonl",
        "finetune_eval_rows.parquet",
    ]:
        src = DOWNLOAD_DIR / "runs" / name
        if not src.exists():
            src = DOWNLOAD_DIR / "artifacts" / "runs" / name
        if src.exists():
            dst = REPO_ROOT / "artifacts" / "runs" / name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            print("Restored:", name)

    for name in [
        "finetune_eval_summary.parquet",
        "finetune_eval_diagnostics.md",
        "finetune_eval_by_language.parquet",
        "finetune_eval_by_label_type.parquet",
    ]:
        src = DOWNLOAD_DIR / "tables" / name
        if not src.exists():
            src = DOWNLOAD_DIR / "artifacts" / "tables" / name
        if src.exists():
            dst = REPO_ROOT / "artifacts" / "tables" / name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            print("Restored:", name)

    print(f"\nAdapter ready at: {adapter_dst}")
    print("Files:", [f.name for f in adapter_dst.iterdir()])
    print("OUTPUT_DIR set to:", OUTPUT_DIR)
    print("\nCells 11-14 (config/train/package/publish-adapter) will be auto-skipped.")
else:
    print("SKIP_TRAINING is False -> will train a fresh adapter in cells 11-14.")

Dataset URL: https://www.kaggle.com/datasets/aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts
License(s): CC0-1.0
0%|          | 0.00/109M [00:00<?, ?B/s]
 10%|█         | 11.0M/109M [00:00<00:00, 107MB/s]
 26%|██▌       | 28.0M/109M [00:00<00:00, 147MB/s]
 39%|███▉      | 43.0M/109M [00:00<00:00, 112MB/s]
 56%|█████▌    | 61.0M/109M [00:00<00:00, 135MB/s]
 76%|███████▌  | 83.0M/109M [00:00<00:00, 146MB/s]
100%|██████████| 109M/109M [00:00<00:00, 159MB/s]
Restored: raw_model_predictions_adapter.jsonl
Restored: tuned_predictions_adapter.jsonl
Restored: finetune_eval_rows.parquet
Restored: finetune_eval_summary.parquet
Restored: finetune_eval_diagnostics.md
Restored: finetune_eval_by_language.parquet
Restored: finetune_eval_by_label_type.parquet

Adapter ready at: /kaggle/working/polyglot_grounded_qa_exports/finetune_unsloth/lora_adapter
Files: ['merges.txt', 'vocab.json', 'adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json', 'added_tokens.json', 'tokenizer.js

In [4]:
# 6) Load T4 config and apply optional overrides
if globals().get("SKIP_TRAINING", False):
    print("SKIP_TRAINING=True -> skipping config load (adapter already pulled).")
else:
    from pathlib import Path
    import yaml

    REPO_ROOT = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
    cfg_path = REPO_ROOT / "configs" / "finetune" / "cloud_unsloth_qlora.yaml"
    CFG = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}

    MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
    MAX_SEQ_LENGTH = 1024
    PER_DEVICE_BATCH = 2
    GRAD_ACCUM = 8
    # With ~4322 train rows and batch=16: ~270 steps/epoch.
    # 540 steps ≈ 2 full epochs on the real XQuAD dataset.
    MAX_STEPS = 540
    SAVE_STEPS = 100
    LOGGING_STEPS = 20
    EVAL_STRATEGY = "no"
    RUN_NAME = "colab-t4-qwen2.5-3b-qlora"

    model_cfg = CFG.get("model", {})
    model_cfg["name"] = MODEL_NAME
    CFG["model"] = model_cfg

    train_cfg = CFG.get("training", {})
    train_cfg["max_seq_length"] = int(MAX_SEQ_LENGTH)
    train_cfg["per_device_train_batch_size"] = int(PER_DEVICE_BATCH)
    train_cfg["gradient_accumulation_steps"] = int(GRAD_ACCUM)
    train_cfg["max_steps"] = int(MAX_STEPS)
    train_cfg["save_steps"] = int(SAVE_STEPS)
    train_cfg["logging_steps"] = int(LOGGING_STEPS)
    train_cfg["eval_strategy"] = EVAL_STRATEGY
    train_cfg["eval_steps"] = int(SAVE_STEPS)
    CFG["training"] = train_cfg
    CFG["run_name"] = RUN_NAME

    print("Loaded", cfg_path)
    print("Run name:", RUN_NAME)
    print("Model:", MODEL_NAME)
    print("Training config:", train_cfg)

Loaded /kaggle/working/polyglot-grounded-qa/configs/finetune/cloud_unsloth_qlora.yaml
Run name: colab-t4-qwen2.5-3b-qlora
Model: Qwen/Qwen2.5-3B-Instruct
Training config: {'max_seq_length': 1024, 'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 8, 'learning_rate': 0.0002, 'warmup_steps': 20, 'max_steps': 540, 'logging_steps': 20, 'save_steps': 100, 'optim': 'adamw_8bit', 'seed': 42, 'eval_strategy': 'no', 'eval_steps': 100}


In [7]:
# 7) Train adapter (output persisted on Drive if mounted)
if globals().get("SKIP_TRAINING", False):
    print("SKIP_TRAINING=True -> skipping training (adapter already pulled).")
else:
    import math
    import os
    os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
    os.environ["DISABLE_TRANSFORMERS_AV"] = "1"
    os.environ["WANDB_DISABLED"] = "true"

    import importlib
    from pathlib import Path
    import yaml

    project_root = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
    if "CFG" in globals():
        cfg = CFG
    else:
        cfg_path = project_root / "configs" / "finetune" / "cloud_unsloth_qlora.yaml"
        cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}

    model_cfg = cfg.get("model", {})
    lora_cfg = cfg.get("lora", {})
    train_cfg = cfg.get("training", {})
    data_cfg = cfg.get("data", {})

    model_name = str(model_cfg.get("name", "Qwen/Qwen2.5-3B-Instruct"))
    max_seq_length = int(train_cfg.get("max_seq_length", 1024))

    train_file = project_root / str(
        data_cfg.get("train", "data/benchmarks/finetune/formatted/train.text.jsonl")
    )
    val_file = project_root / str(
        data_cfg.get("val", "data/benchmarks/finetune/formatted/val.text.jsonl")
    )

    if "DRIVE_RUN_DIR" in globals():
        OUTPUT_DIR = Path(DRIVE_RUN_DIR)
    else:
        OUTPUT_DIR = Path("/content/polyglot_grounded_qa_exports/finetune_unsloth")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    datasets_module = importlib.import_module("datasets")
    trl_module = importlib.import_module("trl")
    transformers_module = importlib.import_module("transformers")
    peft_module = importlib.import_module("peft")
    torch_module = importlib.import_module("torch")

    load_dataset = datasets_module.load_dataset
    SFTConfig = trl_module.SFTConfig
    SFTTrainer = trl_module.SFTTrainer
    AutoModelForCausalLM = transformers_module.AutoModelForCausalLM
    AutoTokenizer = transformers_module.AutoTokenizer
    BitsAndBytesConfig = getattr(transformers_module, "BitsAndBytesConfig", None)
    LoraConfig = peft_module.LoraConfig
    get_peft_model = peft_module.get_peft_model

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {"device_map": "auto", "torch_dtype": torch_module.float16}
    if BitsAndBytesConfig is not None:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    peft_cfg = LoraConfig(
        r=int(lora_cfg.get("rank", 16)),
        lora_alpha=int(lora_cfg.get("alpha", 16)),
        lora_dropout=float(lora_cfg.get("dropout", 0.0)),
        target_modules=list(
            lora_cfg.get(
                "target_modules",
                ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            )
        ),
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_cfg)

    train_ds = load_dataset("json", data_files=str(train_file), split="train")
    val_ds = load_dataset("json", data_files=str(val_file), split="train")

    per_device_batch = int(train_cfg.get("per_device_train_batch_size", 2))
    gradient_accum = int(train_cfg.get("gradient_accumulation_steps", 8))
    max_steps = int(train_cfg.get("max_steps", 300))
    effective_batch = per_device_batch * gradient_accum
    steps_per_epoch = max(1, math.ceil(len(train_ds) / max(effective_batch, 1)))
    approx_epochs = max_steps / steps_per_epoch
    print("Train rows:", len(train_ds))
    print("Val rows:", len(val_ds))
    print("Per-device batch:", per_device_batch)
    print("Gradient accumulation:", gradient_accum)
    print("Effective batch:", effective_batch)
    print("Approx steps/epoch:", steps_per_epoch)
    print(f"Approx epochs at max_steps={max_steps}: {approx_epochs:.2f}")
    if len(train_ds) < 100:
        print("WARNING: very small training set; treat this run as a smoke test or reduce max_steps.")

    eval_strategy = str(train_cfg.get("eval_strategy", "steps"))
    eval_steps = int(train_cfg.get("eval_steps", train_cfg.get("save_steps", 50)))

    sft_args = SFTConfig(
        output_dir=str(OUTPUT_DIR),
        max_seq_length=max_seq_length,
        per_device_train_batch_size=per_device_batch,
        gradient_accumulation_steps=gradient_accum,
        learning_rate=float(train_cfg.get("learning_rate", 2e-4)),
        warmup_steps=int(train_cfg.get("warmup_steps", 20)),
        max_steps=max_steps,
        logging_steps=int(train_cfg.get("logging_steps", 20)),
        save_steps=int(train_cfg.get("save_steps", 100)),
        optim=str(train_cfg.get("optim", "adamw_8bit")),
        seed=int(train_cfg.get("seed", 42)),
        eval_strategy=eval_strategy,
        eval_steps=eval_steps,
        dataset_text_field="text",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=sft_args,
    )

    trainer.train()
    adapter_dir = OUTPUT_DIR / "lora_adapter"
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    print(f"Saved adapter to {adapter_dir}")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/4322 [00:00<?, ? examples/s]

Map:   0%|          | 0/540 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
20,2.445800
40,1.448300
60,1.249000
80,1.180300
100,1.057300
120,1.005300
140,0.952700
160,0.906800
180,0.857100
200,0.799400


Saved adapter to /kaggle/working/polyglot_grounded_qa_exports/finetune_unsloth/lora_adapter


In [8]:
# 8) Verify + package adapter and expose Kaggle-downloadable links
if globals().get("SKIP_TRAINING", False):
    print("SKIP_TRAINING=True -> skipping packaging (adapter already pulled).")
else:
    from pathlib import Path
    import os
    import shutil
    from datetime import UTC, datetime
    from IPython.display import HTML, display

    if "OUTPUT_DIR" not in globals():
        raise RuntimeError("Run the training cell first to define OUTPUT_DIR.")

    IS_KAGGLE = bool(os.getenv("KAGGLE_URL_BASE")) or Path("/kaggle/working").exists()
    WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working") if IS_KAGGLE else Path("/content"))
    adapter_dir = Path(OUTPUT_DIR) / "lora_adapter"
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Missing adapter directory: {adapter_dir}")

    ts = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
    archive_base = Path(WORK_ROOT) / f"lora_adapter_{ts}"
    zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=adapter_dir))

    drive_zip = None
    if "DRIVE_ARTIFACTS_DIR" in globals():
        drive_zip = Path(DRIVE_ARTIFACTS_DIR) / zip_path.name
        if drive_zip != zip_path:
            shutil.copy2(zip_path, drive_zip)

    kaggle_working_zip = Path("/kaggle/working") / zip_path.name
    kaggle_working_zip.parent.mkdir(parents=True, exist_ok=True)
    if kaggle_working_zip != zip_path:
        shutil.copy2(zip_path, kaggle_working_zip)

    cwd_zip = Path.cwd() / zip_path.name
    if cwd_zip != kaggle_working_zip:
        shutil.copy2(kaggle_working_zip, cwd_zip)

    size_mb = kaggle_working_zip.stat().st_size / (1024 * 1024)
    print("Adapter dir:", adapter_dir)
    print("Local zip:", zip_path)
    if drive_zip:
        print("Drive zip:", drive_zip)
    print("Kaggle working zip:", kaggle_working_zip)
    print(f"Zip size: {size_mb:.2f} MB")

    download_name = kaggle_working_zip.name
    display(
        HTML(
            f"""
    <div>
      <p><b>Try these download links (in order):</b></p>
      <ol>
        <li><a href='/files/{download_name}' target='_blank' download>{download_name} via /files route</a></li>
        <li><a href='{download_name}' target='_blank' download>{download_name} relative link</a></li>
        <li><a href='/kaggle/working/{download_name}' target='_blank' download>{download_name} direct working path</a></li>
      </ol>
      <p>If all links are blocked, click <b>Save Version</b> and download from that version's <b>Output files</b>.</p>
    </div>
    """
        )
    )

Adapter dir: /kaggle/working/polyglot_grounded_qa_exports/finetune_unsloth/lora_adapter
Local zip: /kaggle/working/lora_adapter_20260417_213023.zip
Drive zip: /kaggle/working/polyglot_grounded_qa_exports/exports/lora_adapter_20260417_213023.zip
Kaggle working zip: /kaggle/working/lora_adapter_20260417_213023.zip
Zip size: 108.75 MB


In [9]:
# 9) Publish adapter directly as a Kaggle dataset artifact
if globals().get("SKIP_TRAINING", False):
    print("SKIP_TRAINING=True -> skipping adapter publish (already on Kaggle).")
else:
    from pathlib import Path
    import json
    import shutil
    import subprocess
    from datetime import UTC, datetime

    DATASET_ID = "aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts"
    STAGE_ROOT = Path("/kaggle/working/kaggle_artifact_stage")
    ADAPTER_SRC = Path(OUTPUT_DIR) / "lora_adapter" if "OUTPUT_DIR" in globals() else None

    if ADAPTER_SRC is None or not ADAPTER_SRC.exists():
        raise FileNotFoundError("Adapter directory missing. Run the training cell first.")

    adapter_dst = STAGE_ROOT / "artifacts" / "runs" / "finetune_unsloth" / "lora_adapter"
    if adapter_dst.exists():
        shutil.rmtree(adapter_dst)
    adapter_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(ADAPTER_SRC, adapter_dst)

    zip_candidates = sorted(Path("/kaggle/working").glob("lora_adapter_*.zip"))
    if zip_candidates:
        exports_dir = STAGE_ROOT / "artifacts" / "exports"
        exports_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zip_candidates[-1], exports_dir / zip_candidates[-1].name)

    metadata = {
        "title": "polyglot-grounded-qa-finetune-artifacts",
        "id": DATASET_ID,
        "licenses": [{"name": "Apache-2.0"}],
    }
    metadata_path = STAGE_ROOT / "dataset-metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    def run_cmd(cmd: list[str]) -> subprocess.CompletedProcess[str]:
        print("Running:", " ".join(cmd))
        cp = subprocess.run(cmd, text=True, capture_output=True)
        if cp.stdout.strip():
            print(cp.stdout.strip())
        if cp.stderr.strip():
            print(cp.stderr.strip())
        return cp

    status_cp = run_cmd(["kaggle", "datasets", "status", DATASET_ID])
    message = f"sync lora adapter {datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S UTC')}"

    if status_cp.returncode == 0:
        publish_cp = run_cmd([
            "kaggle", "datasets", "version",
            "-p", str(STAGE_ROOT),
            "-m", message,
            "--dir-mode", "zip",
        ])
    else:
        publish_cp = run_cmd([
            "kaggle", "datasets", "create",
            "-p", str(STAGE_ROOT),
            "--dir-mode", "zip",
        ])

    if publish_cp.returncode != 0:
        raise RuntimeError(
            "Kaggle publish failed. If credentials are missing, set KAGGLE_USERNAME and KAGGLE_KEY "
            "in Kaggle notebook Secrets (do not paste them in chat)."
        )

    print("Published dataset artifact:", DATASET_ID)
    print("Staged adapter path:", adapter_dst)
    print("Stage root:", STAGE_ROOT)

Running: kaggle datasets status aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts
ready
Running: kaggle datasets version -p /kaggle/working/kaggle_artifact_stage -m sync lora adapter 2026-04-17 21:30:37 UTC --dir-mode zip
Starting upload for file artifacts.zip
Upload successful: artifacts.zip (326MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts
0%|          | 0.00/326M [00:00<?, ?B/s]
  2%|▏         | 6.27M/326M [00:00<00:05, 65.3MB/s]
  8%|▊         | 25.4M/326M [00:00<00:02, 145MB/s] 
 12%|█▏        | 39.3M/326M [00:00<00:02, 119MB/s]
 16%|█▌        | 52.5M/326M [00:00<00:02, 115MB/s]
 22%|██▏       | 71.0M/326M [00:00<00:02, 129MB/s]
 26%|██▌       | 83.5M/326M [00:00<00:02, 110MB/s]
 29%|██▉       | 95.1M/326M [00:00<00:02, 109MB/s]
 34%|███▍      | 111M/326M [00:00<00:01, 125MB/s] 
 38%|███▊      | 124M/326M [00:01<00:02, 99.8MB/s]
 42%|████▏     | 137M/326M [00:01<00:01, 105MB/

In [10]:
# 10) Run trained-adapter eval append on Kaggle runtime
from pathlib import Path
import gc
import os
import shutil
import subprocess
import sys

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
repo_adapter_dir = repo_root / "artifacts" / "runs" / "finetune_unsloth" / "lora_adapter"
raw_pred = repo_root / "artifacts" / "runs" / "raw_model_predictions_adapter.jsonl"
norm_pred = repo_root / "artifacts" / "runs" / "tuned_predictions_adapter.jsonl"

# Release training objects before launching adapter inference in a subprocess.
for name in ["trainer", "model", "tokenizer", "train_ds", "val_ds"]:
    if name in globals():
        del globals()[name]
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception as exc:
    print("CUDA cleanup warning:", exc)

candidate_dirs = []
if "OUTPUT_DIR" in globals():
    candidate_dirs.append(Path(OUTPUT_DIR) / "lora_adapter")
candidate_dirs.append(Path("/content/polyglot_grounded_qa_exports/finetune_unsloth/lora_adapter"))

adapter_src = next((p for p in candidate_dirs if p.exists()), None)
if adapter_src is None:
    raise FileNotFoundError("Could not find adapter dir in OUTPUT_DIR or /content/polyglot_grounded_qa_exports.")

if repo_adapter_dir.exists():
    shutil.rmtree(repo_adapter_dir)
repo_adapter_dir.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(adapter_src, repo_adapter_dir)
print("Synced adapter into repo path:", repo_adapter_dir)

def run_repo_cmd(cmd: list[str]) -> None:
    print("Running:", " ".join(cmd))
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    src_path = str(repo_root / "src")
    env["PYTHONPATH"] = src_path + os.pathsep + env.get("PYTHONPATH", "")
    cp = subprocess.run(cmd, cwd=repo_root, env=env, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout.strip())
    if cp.stderr.strip():
        print(cp.stderr.strip())
    if cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {' '.join(cmd)}")

py = sys.executable
run_repo_cmd([
    py, "scripts/generate_tuned_predictions.py",
    "--mode", "hf-adapter",
    "--test-file", "data/benchmarks/finetune/test.jsonl",
    "--base-model", "Qwen/Qwen2.5-3B-Instruct",
    "--adapter-path", str(repo_adapter_dir),
    "--output", str(raw_pred),
    "--max-new-tokens", "192",
    "--temperature", "0.0",
])
run_repo_cmd([
    py, "scripts/normalize_tuned_predictions.py",
    "--input", str(raw_pred),
    "--output", str(norm_pred),
])
run_repo_cmd([
    py, "scripts/run_finetune_eval.py",
    "--variant", "tuned-adapter-v1",
    "--predictions", str(norm_pred),
    "--test-file", "data/benchmarks/finetune/test.jsonl",
    "--append",
])
print("Kaggle-side tuned-adapter eval append completed.")

Synced adapter into repo path: /kaggle/working/polyglot-grounded-qa/artifacts/runs/finetune_unsloth/lora_adapter
Running: /usr/bin/python3 scripts/generate_tuned_predictions.py --mode hf-adapter --test-file data/benchmarks/finetune/test.jsonl --base-model Qwen/Qwen2.5-3B-Instruct --adapter-path /kaggle/working/polyglot-grounded-qa/artifacts/runs/finetune_unsloth/lora_adapter --output /kaggle/working/polyglot-grounded-qa/artifacts/runs/raw_model_predictions_adapter.jsonl --max-new-tokens 192 --temperature 0.0
Wrote 539 predictions to /kaggle/working/polyglot-grounded-qa/artifacts/runs/raw_model_predictions_adapter.jsonl

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.22s/it]
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnin

In [ ]:
# 10.5) Run prompted base-model comparison on the same test set
from pathlib import Path
import os
import subprocess
import sys

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
raw_pred = repo_root / "artifacts" / "runs" / "raw_model_predictions_base_prompted.jsonl"
norm_pred = repo_root / "artifacts" / "runs" / "tuned_predictions_base_prompted.jsonl"

def run_repo_cmd(cmd: list[str]) -> None:
    print("Running:", " ".join(cmd))
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    src_path = str(repo_root / "src")
    env["PYTHONPATH"] = src_path + os.pathsep + env.get("PYTHONPATH", "")
    cp = subprocess.run(cmd, cwd=repo_root, env=env, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout.strip())
    if cp.stderr.strip():
        print(cp.stderr.strip())
    if cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {' '.join(cmd)}")

py = sys.executable
run_repo_cmd([
    py, "scripts/run_trained_adapter_eval.py",
    "--variant", "base-model-prompted-v1",
    "--no-adapter",
    "--test-file", "data/benchmarks/finetune/test.jsonl",
    "--raw-output", str(raw_pred),
    "--normalized-output", str(norm_pred),
    "--append",
])
print("Prompted base-model comparison completed.")

Running: /usr/bin/python3 scripts/run_trained_adapter_eval.py --variant base-model-prompted-v1 --no-adapter --test-file data/benchmarks/finetune/test.jsonl --raw-output /kaggle/working/polyglot-grounded-qa/artifacts/runs/raw_model_predictions_base_prompted.jsonl --normalized-output /kaggle/working/polyglot-grounded-qa/artifacts/runs/tuned_predictions_base_prompted.jsonl --append


In [14]:
# Debug: check raw predictions are different
import json
def read_jsonl(p):
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]

raw_adapter = read_jsonl(REPO_ROOT / "artifacts" / "runs" / "raw_model_predictions_adapter.jsonl")
raw_base = read_jsonl(REPO_ROOT / "artifacts" / "runs" / "raw_model_predictions_base_prompted.jsonl")
norm_adapter = read_jsonl(REPO_ROOT / "artifacts" / "runs" / "tuned_predictions_adapter.jsonl")
norm_base = read_jsonl(REPO_ROOT / "artifacts" / "runs" / "tuned_predictions_base_prompted.jsonl")

print(f"Raw adapter: {len(raw_adapter)}, Raw base: {len(raw_base)}")
print(f"Norm adapter: {len(norm_adapter)}, Norm base: {len(norm_base)}")

# Check if generated_text differs
same = sum(1 for a, b in zip(raw_adapter, raw_base) if a.get("generated_text") == b.get("generated_text"))
print(f"Same generated_text: {same}/{len(raw_adapter)}")

# Check normalized answers
nonempty_adapter = sum(1 for r in norm_adapter if r.get("answer", "").strip())
nonempty_base = sum(1 for r in norm_base if r.get("answer", "").strip())
print(f"Non-empty adapter answers: {nonempty_adapter}/{len(norm_adapter)}")
print(f"Non-empty base answers: {nonempty_base}/{len(norm_base)}")

# Show a sample
print("\n--- Adapter row 0 ---")
print("  generated_text[:200]:", raw_adapter[0].get("generated_text","")[:200])
print("  norm answer:", norm_adapter[0].get("answer","")[:100] if norm_adapter else "N/A")
print("\n--- Base row 0 ---")
print("  generated_text[:200]:", raw_base[0].get("generated_text","")[:200])
print("  norm answer:", norm_base[0].get("answer","")[:100] if norm_base else "N/A")

Raw adapter: 539, Raw base: 539
Norm adapter: 398, Norm base: 398
Same generated_text: 539/539
Non-empty adapter answers: 398/398
Non-empty base answers: 398/398

--- Adapter row 0 ---
  generated_text[:200]: {"raw": "Citations:\n[57296d1b1d0469140077940e-chunk-0]\nAnswer: konak h\u00fccre zar\u0131n\u0131n \u00fcr\u00fcn\u00fc anlam\u0131na geldi\u011fi \u015feklinde yorumlan\u0131r\u2014ki bu yanl\u0131\
  norm answer: konak hücre zarının ürünü anlamına geldiği şeklinde yorumlanır—ki bu yanlıştır

--- Base row 0 ---
  generated_text[:200]: {"raw": "Citations:\n[57296d1b1d0469140077940e-chunk-0]\nAnswer: konak h\u00fccre zar\u0131n\u0131n \u00fcr\u00fcn\u00fc anlam\u0131na geldi\u011fi \u015feklinde yorumlan\u0131r\u2014ki bu yanl\u0131\
  norm answer: konak hücre zarının ürünü anlamına geldiği şeklinde yorumlanır—ki bu yanlıştır


In [12]:
# 11) Publish refreshed full artifacts tree to Kaggle dataset
from pathlib import Path
import json
import shutil
import subprocess
from datetime import UTC, datetime

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
dataset_id = "aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts"
stage_root = Path("/kaggle/working/kaggle_artifact_stage_full")
src_artifacts = repo_root / "artifacts"
dst_artifacts = stage_root / "artifacts"

if not src_artifacts.exists():
    raise FileNotFoundError(f"Missing artifacts directory: {src_artifacts}")

if dst_artifacts.exists():
    shutil.rmtree(dst_artifacts)
dst_artifacts.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src_artifacts, dst_artifacts)

metadata = {
    "title": "polyglot-grounded-qa-finetune-artifacts",
    "id": dataset_id,
    "licenses": [{"name": "Apache-2.0"}],
}
(stage_root / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

def run_publish_cmd(cmd: list[str]) -> subprocess.CompletedProcess[str]:
    print("Running:", " ".join(cmd))
    cp = subprocess.run(cmd, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout.strip())
    if cp.stderr.strip():
        print(cp.stderr.strip())
    return cp

status_cp = run_publish_cmd(["kaggle", "datasets", "status", dataset_id])
message = f"sync full artifacts {datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S UTC')}"

if status_cp.returncode == 0:
    publish_cp = run_publish_cmd([
        "kaggle", "datasets", "version",
        "-p", str(stage_root),
        "-m", message,
        "--dir-mode", "zip",
    ])
else:
    publish_cp = run_publish_cmd([
        "kaggle", "datasets", "create",
        "-p", str(stage_root),
        "--dir-mode", "zip",
    ])

if publish_cp.returncode != 0:
    raise RuntimeError("Failed to publish full artifacts dataset version.")

print("Published refreshed artifacts to:", dataset_id)
print("Staged from:", src_artifacts)

Running: kaggle datasets status aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts
ready
Running: kaggle datasets version -p /kaggle/working/kaggle_artifact_stage_full -m sync full artifacts 2026-04-17 23:29:54 UTC --dir-mode zip
Starting upload for file artifacts.zip
Upload successful: artifacts.zip (109MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts
0%|          | 0.00/109M [00:00<?, ?B/s]
  7%|▋         | 8.19M/109M [00:00<00:01, 71.8MB/s]
 21%|██        | 23.0M/109M [00:00<00:00, 116MB/s] 
 31%|███▏      | 34.4M/109M [00:00<00:00, 98.2MB/s]
 40%|████      | 44.2M/109M [00:00<00:00, 90.4MB/s]
 49%|████▉     | 53.6M/109M [00:00<00:00, 92.4MB/s]
 57%|█████▋    | 62.6M/109M [00:00<00:00, 87.3MB/s]
 71%|███████▏  | 77.9M/109M [00:00<00:00, 106MB/s] 
 81%|████████  | 88.2M/109M [00:00<00:00, 101MB/s]
 95%|█████████▌| 104M/109M [00:01<00:00, 99.7MB/s]
100%|██████████| 109M/109M [00:01<0

## Local follow-up (back in VS Code)

1. Pull artifacts: `kaggle datasets download aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts -p /tmp/kaggle_artifacts --unzip`
2. Copy adapter to `artifacts/runs/finetune_unsloth/lora_adapter` and eval files to `artifacts/tables/` + `artifacts/runs/`.
3. Run `uv run python scripts/check_final_artifacts_contract.py --core-only` to verify.
4. Re-run comparison cells in `notebooks/80_final_results.ipynb`.

### Dataset scale (current)

| Split | Rows | Notes |
|-------|------|-------|
| Train | ~4322 | Real XQuAD (en/es/tr) + hard negatives |
| Val | ~540 | Language-stratified |
| Test | ~539 | 23 hard negatives + 38 synthetic negatives |

### Run order (top to bottom)

**Path A — Inference only (SKIP_TRAINING=True, ~30 min):**
1. Cells 2-9: environment setup, repo clone, deps, data prep
2. Cell 10: pull adapter from Kaggle dataset
3. Cells 11-14: auto-skip (config/train/package/publish)
4. Cells 15-16: adapter eval + base model comparison
5. Cell 17: publish full artifacts

**Path B — Full training (~4 h on T4x2):**
1. Cells 2-9: environment setup, repo clone, deps, data prep
2. Cell 10: set `SKIP_TRAINING = False` (default)
3. Cells 11-14: config (MAX_STEPS=540), train, package, publish adapter
4. Cells 15-16: adapter eval + base model comparison on 539-row test set
5. Cell 17: publish full artifacts